**Running this on Google Colab.** The first code cell fetches this entry's files and installs the packages it was run with, pinned for Colab's Python version. The results in the registry come from the entry's own environment ([requirements.lock.txt](requirements.lock.txt), Python 3.13); a different Python or platform can change the last digits of some numbers.

In [ ]:
# On Google Colab: fetch this entry's files and install its pinned packages. Elsewhere this cell does nothing.
import sys
if "google.colab" in sys.modules:
    import os, subprocess
    if not os.path.isdir("/content/encore"):
        subprocess.run(["git", "clone", "-q", "--depth", "1", "--branch", "colab", "--filter=blob:none", "--sparse",
                        "https://github.com/encorescience/encore-registry", "/content/encore"], check=True)
        subprocess.run(["git", "-C", "/content/encore", "sparse-checkout", "set", "entries/garud-2026"], check=True)
    os.chdir("/content/encore/entries/garud-2026")
    sys.path.insert(0, os.getcwd())
    version = f"{sys.version_info.major}.{sys.version_info.minor}"
    lock = f"requirements.colab-{version}.txt"
    if not os.path.exists(lock):
        print(f"No lock for Python {version}: installing the entry's unpinned requirements (requirements.in).")
        lock = "requirements.in"
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
    subprocess.run(["uv", "pip", "install", "--system", "-q", "-r", lock], check=True)
    print("Installed. If Colab asks to restart the session, restart it and run the notebook from the top again.")


# Reproducing Garud & Morris (2026) on DANDI:001552 and the Krausz et al. (2023) table

**Paper.** Garud S, Morris L (2026). Continuous Temporal Difference Learning as a Unifying Theory
of Dopamine Function. bioRxiv preprint,
[doi:10.64898/2026.04.07.716149](https://doi.org/10.64898/2026.04.07.716149), version posted April
8, 2026. The claims file was written from this preprint's text (main text, legends and Methods);
the supplement was not available.

**Dataset.** [DANDI:001552](https://dandiarchive.org/dandiset/001552/0.250828.1728), version
0.250828.1728 (Floeder et al. 2025), streamed from the archive. This is the version the paper's
Data Availability cites, so it is pinned. A later version, 0.250903.1802, holds the same assets
(same paths, asset ids and sizes; data check), so the choice does not change the analysis.

**Outside data.** The paper's Figures 2, 3 and 5 use the Krausz et al. (2023) hex-level table,
which the paper cites at Mendeley Data
[doi:10.17632/m59zdjpm9h.1](https://doi.org/10.17632/m59zdjpm9h.1). Its file hexLevelDf.csv.zip is
read into memory and checked against its SHA-256. These claims are `data: elsewhere` and are
counted apart from the dandiset.

**Random draws.** No analysis here uses random numbers, so no seed is set.

| Claim | Data | Tested |
|---|---|---|
| floeder-subjects-sessions | DANDI:001552 | yes |
| fig4a-lick-short-above-long | DANDI:001552 | yes |
| fig4b-long-iti-dip-below-baseline | DANDI:001552 | yes |
| fig4d-dip-deepens-with-training | DANDI:001552 | yes |
| fig4c-lick-rate-rises-with-training | DANDI:001552 | yes |
| krausz-included-observations | Mendeley hexLevelDf | yes |
| fig2a-ramp-toward-goal | Mendeley hexLevelDf | yes |
| fig2b-ramp-scales-with-reward | Mendeley hexLevelDf | yes |
| fig2c-ramp-similar-early-late | Mendeley hexLevelDf | yes |
| fig2d-ramp-scales-with-velocity | Mendeley hexLevelDf | yes |
| fig2e-velocity-covaries-with-distance | Mendeley hexLevelDf | yes |
| fig2f-coupling-grows-near-goal | Mendeley hexLevelDf | yes |
| fig2gh-stationary-dip | Mendeley hexLevelDf | yes |
| fig3-recovered-value-increases | Mendeley hexLevelDf | yes |
| fig3cd-gaussian-shape-wins | Mendeley hexLevelDf | yes |
| fig5bc-gaussian-ct-td-best | Mendeley hexLevelDf | yes |
| fig5d-tonic-term-and-gaussian-contribute | Mendeley hexLevelDf | yes |
| figs2-no-velocity-coupling-without-goal | Brown et al. (2026) | blocked: the supplement holding its method was not available |

In [ ]:
import time
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from scipy import stats

import ctd

warnings.filterwarnings("ignore")
pd.set_option("display.width", 220)
pd.set_option("display.max_columns", 40)
pd.set_option("display.max_rows", 400)

started = time.time()
assets = ctd.session_table(ctd.list_assets(ctd.VERSION))
later = ctd.list_assets("0.250903.1802")
with_events = assets[assets.group != "VR-FP"].reset_index(drop=True)
all_summaries = ctd.summarise_all(with_events)
T_all = ctd.session_frame(with_events, all_summaries)
# a recording deposited twice (identical 470 trace) is counted once, at its first day
keep = ~T_all.duplicated("hash470", keep="first").to_numpy()
summaries = [s for s, k in zip(all_summaries, keep) if k]
with_events_all, with_events = with_events, with_events[keep].reset_index(drop=True)
T = ctd.session_frame(with_events, summaries)
vr = assets[assets.group == "VR-FP"]
from concurrent.futures import ThreadPoolExecutor
with ThreadPoolExecutor(8) as ex:
    vr_contents = list(ex.map(ctd.file_contents, vr.asset_id))
contents_one = {g: ctd.file_contents(with_events[with_events.group == g].asset_id.iloc[0])
                for g in ("FP", "dLight-ramp2")}
hexdf = ctd.load_hex()
inc = ctd.included(hexdf)
I = hexdf[inc].copy()
print(f"dandiset {ctd.DANDISET} version {ctd.VERSION}: {len(assets)} files; "
      f"{len(with_events)} sessions read in full; hexLevelDf {len(hexdf)} rows; "
      f"{time.time() - started:.0f} s")

## Data check

Each check of the metadata against the values is stated under its output.

In [ ]:
print("Files per subject and group:")
print(assets.groupby(["group", "subject"]).size().to_string())
print("\nVersions 0.250828.1728 and 0.250903.1802: same paths",
      set(assets.path) == set(later.path), "| same asset ids",
      set(assets.asset_id) == set(later.asset_id), "| same sizes",
      assets.set_index("path")["size"].equals(later.set_index("path")["size"].loc[assets.path]))
print("\nSession descriptions:", T.session_description.value_counts().to_dict())
vr_acq = pd.Series([", ".join(c["acquisition"]) for c in vr_contents]).value_counts()
print("VR-FP files by acquisition contents:", vr_acq.to_dict(),
      "| with any processing module:", sum(bool(c["processing"]) for c in vr_contents), "of", len(vr_contents))
for g, c in contents_one.items():
    print(f"first {g} file: acquisition {c['acquisition']}; processing {c['processing']}")

The dandiset holds 27 mice. All 144 files of the 9 `JF-VR-FP` mice hold only the raw photometry
and TTL channels, with no event log and no processing module, so they cannot be used for any
Figure 4 measure; the 18 `JF-FP` and `JF-dLight-ramp2` mice have photometry and the event log,
which matches the paper's n = 18. In these files `processing/` holds only the task parameters, so
there is no depositor ΔF/F; ΔF/F is formed here from the raw 470 and 405 nm channels. The two
published versions hold the same assets, so the choice of version does not matter for the
analysis.

In [ ]:
per = T.groupby("subject").agg(group=("group", "first"), sessions=("day", "size"),
                               long=("cls", lambda c: (c == "long").sum()),
                               short=("cls", lambda c: (c == "short").sum()),
                               first_day=("day", "min"), last_day=("day", "max"))
per["order"] = T.sort_values(["subject", "day"]).groupby("subject").cls.apply(
    lambda c: "".join("L" if x == "long" else "S" for x in c))
print(per.to_string())
print("\nMedian inter-cue interval by class (s):")
print(T.groupby("cls").median_ici.describe().round(2).to_string())
print("\nCues per session by class:", T.groupby("cls").n_cues.agg(["min", "median", "max"]).to_dict())
print("Rewards per session by class:", T.groupby("cls").n_rewards.agg(["min", "median", "max"]).to_dict())
lags = np.concatenate([s["cue_reward_lag"] for s in summaries])
print(f"Cue to reward (s): median {np.median(lags):.3f}, range {lags.min():.3f} to {lags.max():.3f}; "
      f"cues with no reward within 10 s: {int((T.n_trials - T.n_rewarded).sum())}")
print("Median ITI from reward to next cue by class (s):",
      T.groupby("cls").median_iti_reward.median().round(2).to_dict())
print("Licks per session:", T.n_licks.min(), "to", T.n_licks.max())

Sessions come in blocks, not alternating day by day: the `JF-FP` mice have 8 long-ITI sessions
then 8 short-ITI sessions, and the `JF-dLight-ramp2` mice 7 or 8 long, 8 short, then 7 or 8 long
(counting each recording once). The
paper's "alternating ... across days" does not describe this order. Rewards follow the cue by
9 s, as `CS_t_fxd` = 9000 ms in the task parameters says.

In [ ]:
clk = T[["subject", "day", "group", "stated_rate", "clock_a", "clock_resid", "clock_pairs",
         "n_edges", "n_cues", "n_samples", "n_nan470"]].copy()
clk["fitted_rate_hz"] = 1 / clk.clock_a
print("Stated photometry rate (Hz) by group:")
print(clk.groupby("group").stated_rate.agg(["min", "max"]).to_string())
print("\nRate fitted from TTL2 edges against cue times (Hz), by stated rate:")
print(clk.groupby(["group", clk.stated_rate.round(1)]).fitted_rate_hz.describe().round(3).to_string())
print("\nLargest clock residual over the pairs kept (s): max", clk.clock_resid.max().round(4),
      "| sessions with residual above 0.05 s:", int((clk.clock_resid > 0.05).sum()))
print("Sessions where TTL2 edges and cues differ in number:")
print(clk[clk.n_edges != clk.n_cues].to_string(index=False))
print("Sessions with a pair left out of the fit (more than 0.1 s off):",
      int((clk.clock_pairs < np.minimum(clk.n_edges, clk.n_cues)).sum()))
print("\nPhotometry span on the event clock against the last event (s), first 5 sessions:")
print(pd.DataFrame([{"start": s["phot_span"][0], "end": s["phot_span"][1], "last_event": s["event_end"]}
                    for s in summaries[:5]]).round(1).to_string())
starts = np.array([s["phot_span"][0] for s in summaries])
print("photometry start on the event clock, all sessions (s): min", starts.min().round(1), "max", starts.max().round(1))
print("stated start of the photometry series (starting_time, or the first timestamp), all sessions (s): min",
      T.stated_start.min(), "max", T.stated_start.max())
print("description of photometry_ttl2, all sessions:", sorted(T.ttl2_description.unique()))
print("\nNaN samples in 470 per session: max", T.n_nan470.max(), "| sessions with any:", int((T.n_nan470 > 0).sum()))

The stated rate is wrong in part of the `JF-FP` files: 64 of them state 120481.9 Hz (or
timestamps 8.3 microseconds apart), and the TTL2 channel gives 120.29 to 120.53 Hz (mean 120.41),
a factor of about 1000. The other files state 130 Hz, which the TTL2 channel confirms. Every
photometry series states a start of 0 (at most 8 microseconds), and the event log has its own
zero. The only link between them is `photometry_ttl2`, described in every file only as "raw
photometry channel ttl2". Each TTL2 rising edge falls on a cue onset of the event log, within
0.1 s once one outlying pair is left out in 6 sessions, so the photometry can be placed on the
event clock; on it, the photometry starts 3.6 to 251 s before the event log's zero.

In [ ]:
con = pd.DataFrame({"cls": T.cls, "true": T.contrast_true,
                    "shifted_mean": [np.mean(s["contrast_shift"]) for s in summaries]})
print("Reward response (ΔF/F 0-2 s after minus 2-0 s before reward), true times against times "
      "shifted by -6, -3, +3, +6 s (mean of the four):")
print(con.groupby("cls")[["true", "shifted_mean"]].describe().round(4).T.to_string())
long_c = con[con.cls == "long"]
print("Long-ITI sessions, true > shifted:", int((long_c.true > long_c.shifted_mean).sum()), "of", len(long_c),
      "| Wilcoxon p =", f"{stats.wilcoxon(long_c.true, long_c.shifted_mean).pvalue:.2g}")
print("ΔF/F (fit reading) range over sessions:",
      round(min(s["dff_range"][0] for s in summaries), 3), "to", round(max(s["dff_range"][1] for s in summaries), 3),
      "| 0.1th and 99.9th percentiles within a session, extremes over sessions:",
      round(min(s["dff_pct"][0] for s in summaries), 3), "and", round(max(s["dff_pct"][2] for s in summaries), 3))
low = T[T.n_low470 > 0]
print("sessions with 470 samples below 0.01 (a signal near zero):", len(low), "| samples:",
      low.set_index(["subject", "day"]).n_low470.to_dict())
print("Raw 470 range:", round(min(s["range470"][0] for s in summaries), 4), "to",
      round(max(s["range470"][1] for s in summaries), 4), "| 405:",
      round(min(s["range405"][0] for s in summaries), 4), "to", round(max(s["range405"][1] for s in summaries), 4),
      "(arbitrary units, as stated)")
dup470 = T_all[T_all.duplicated("hash470", keep=False)]
print("\nFiles with identical 470 traces (compared over the whole trace):")
print(dup470[["path", "n_samples", "n_cues", "n_licks", "hash470"]].to_string(index=False))
dupcue = T_all[T_all.duplicated("hash_cues", keep=False) & ~T_all.duplicated("hash470", keep=False)]
print("\nOther files sharing their sequence of inter-cue intervals with another file:")
print(dupcue[["path", "n_cues", "n_licks", "n_samples", "hash_cues"]].to_string(index=False))
print("\nsessions analysed after counting each recording once:", len(T), "of", len(T_all))
cue_tr = pd.DataFrame([s["cue_trace"] for s in summaries], columns=np.arange(-5, 15))
print("\nCue-locked ΔF/F (fit reading), mean over sessions, 1 s bins starting at the second shown "
      "(cue at 0, reward at 9):")
print(cue_tr.groupby(T.cls.values).mean().round(4).to_string())
print("short-ITI sessions with the 0-1 s bin above the -2 to 0 s mean:",
      int((cue_tr[0] > cue_tr[[-2, -1]].mean(1))[T.cls.values == "short"].sum()), "of", int((T.cls == "short").sum()),
      "| with the 9-10 s bin above the 5-7 s mean:",
      int((cue_tr[9] > cue_tr[[5, 6]].mean(1))[T.cls.values == "short"].sum()))

The extreme ΔF/F values come from short stretches where both channels fall near zero (listed
above), where the fitted 405 signal is near zero too; they are kept, as the paper names no
exclusion, and they touch few 1 s bins.

With the fitted clock, the reward response of long-ITI sessions is larger at the true reward
times than at shifted ones, which supports the alignment. In short-ITI sessions the shifted
windows land on other trial events, so that contrast cannot tell; the cue-locked trace does. In
both classes ΔF/F rises sharply in the first second after the cue, from a flat level before it,
and rises again from 8 to 10 s around the reward at 9 s (counts above), so the short-ITI sessions
are aligned too.

Two recordings are deposited twice: in `JF-dLight-ramp2-F2` the Day3 and Day4 files, and in
`JF-dLight-ramp2-F4` the Day5 and Day6 files, hold identical photometry, cues and licks. Each is
counted once, at its first day, and the second file is left out (sessions are then numbered
without it). Other files share a cue schedule with another mouse's session but hold different
photometry and licks; these are a reused trial schedule, not duplicates.

In [ ]:
print("hexLevelDf columns:", list(hexdf.columns))
print("rows", len(hexdf), "| rats", hexdf.rat.nunique(), "| sessions", hexdf.session.nunique())
print("rows by rat:", hexdf.rat.value_counts().sort_index().to_dict())
print("sessions by rat:", hexdf.groupby("rat").session.nunique().to_dict())
print("\nport values:", hexdf.port.value_counts().to_dict(), "| rwd:", hexdf.rwd.value_counts().to_dict())
print("hexesFromPort: missing", int(hexdf.hexesFromPort.isna().sum()), "| zero", int((hexdf.hexesFromPort == 0).sum()),
      "| range", hexdf.hexesFromPort.min(), "to", hexdf.hexesFromPort.max())
print("rows at a port with hexesFromPort 0:", int(((hexdf.port != -100) & (hexdf.hexesFromPort == 0)).sum()),
      "| at a port, missing:", int(((hexdf.port != -100) & hexdf.hexesFromPort.isna()).sum()))
print("vel: missing", int(hexdf.vel.isna().sum()), "| percentiles 0/1/50/99/100:",
      np.nanpercentile(hexdf.vel, [0, 1, 50, 99, 100]).round(2))
print("DA percentiles 0/1/50/99/100:", np.percentile(hexdf.DA, [0, 1, 50, 99, 100]).round(3),
      "| mean", round(hexdf.DA.mean(), 4), "sd", round(hexdf.DA.std(), 4))
print("destination reward of included rows (nom_rwd of the port the trial ends at):",
      I.dest_rwd.value_counts().sort_index().to_dict())

The table matches the Methods' description: 298,918 rows, 10 rats, 82 sessions (5 to 12 per
rat). Units are not stated in the file; velocity values (median about 32) are consistent with
cm/s and DA with z-scores (mean near 0, SD 0.72), as the Mendeley description says. `port` is -100
except at the row where a trial ends at a port, and `hexesFromPort` is missing on each trial's
first row (the start hex) and 0 at the port. The nominal rewards take the values 10, 20, 50, 80 and
90. The paper calls them µL; the Krausz et al. code (github.com/Berke-lab/DA_maze, commit 27eb743,
`DA/Modules/hexLevelAnalyses.py`) divides `nom_rwd_chosen` by 100 and calls it pRwd, so they are
reward probabilities in percent. The split at 80 and 20 is the same either way.

## floeder-subjects-sessions

> The Floeder dataset holds 18 mice, each with 16 to 24 sessions, alternating between short-ITI
> and long-ITI conditions across days.

In [ ]:
n_mice = T.subject.nunique()
counts = T.groupby("subject").size()
both = T.groupby("subject").cls.nunique() == 2
print("mice with photometry and events:", n_mice)
print("sessions per mouse: min", counts.min(), "max", counts.max(), "| all in 14.4 to 26.4:",
      bool(counts.between(14.4, 26.4).all()))
print("both classes in every mouse:", bool(both.all()))
print("session median inter-cue interval, median by class (s):", T.groupby("cls").median_ici.median().round(1).to_dict(),
      "| against Results ~8 and ~55 s, Methods ~15 and ~45 s")
print("mice without events (VR-FP):", vr.subject.nunique())

**Result: reproduced.** 18 mice have photometry and events, with 16 to 24 sessions each, and every
mouse has both classes. The class medians of the inter-cue interval are about 16.6 s and 53.7 s, so
the short value fits the Methods' ~15 s and the long value the Results' ~55 s. The sessions are in
blocks, not alternating day by day (data check).

## fig4a-lick-short-above-long

> Mice show higher anticipatory lick rates in short-ITI sessions than in long-ITI sessions.

In [ ]:
lick_a = T.groupby(["subject", "cls"]).lick_rate.mean().unstack()
pooled = []
for (_, r), s in zip(with_events.iterrows(), summaries):
    c = "short" if s["median_ici"] < ctd.SHORT_LONG_SPLIT else "long"
    pooled += [(r.subject, c, x) for x in s["lick_rates"]]
lick_b = pd.DataFrame(pooled, columns=["subject", "cls", "rate"]).groupby(["subject", "cls"]).rate.mean().unstack()
for name, tab in [("(a) mean of session means", lick_a), ("(b) trials pooled", lick_b)]:
    tt = stats.ttest_rel(tab.short, tab.long)
    print(f"{name}: short {tab.short.mean():.3f} Hz, long {tab.long.mean():.3f} Hz, "
          f"difference {np.mean(tab.short - tab.long):.3f} Hz, paired t({len(tab) - 1}) = {tt.statistic:.2f}, "
          f"p = {tt.pvalue:.2g}; short higher in {(tab.short > tab.long).sum()} of {len(tab)} mice")
print("\nPer mouse, reading (a), Hz:")
print(lick_a.round(3).assign(group=T.groupby("subject").group.first()).to_string())
print("range of per-mouse rates:", lick_a.min().min().round(3), "to", lick_a.max().max().round(3))

**Result: reproduced.** Under both readings the anticipatory lick rate is higher in short-ITI
sessions (2.39 against 1.60 Hz, paired t(17) = 4.15, p = 0.00067), in 16 of 18 mice.

## fig4b-long-iti-dip-below-baseline

> During the late inter-trial interval of long-ITI sessions, dopamine settles below baseline,
> while in short-ITI sessions a slowly decaying reward response does not resolve before the next
> trial.

Eight readings: correction (fit, own) x ITI reference (reward, cue) x observation (trials,
samples). The long-ITI test is one-sample, one-tailed, across mice of each mouse's mean tonic
measure; the short-ITI part is the last kept bin before the next cue. Of the ITI references, the
paper best supports the one from reward: the median time from reward to the next cue is 7.6 s in
short and 44.7 s in long sessions (data check), the Results' ~8 s and the Methods' ~45 s, while
from cue to cue the long value is 53.7 s. The reward reading is adopted where one is needed; all
are shown.

Baseline is ΔF/F of zero (criterion). Both corrections regress the 405 nm signal over the whole
session, so the mean ΔF/F of a session is close to zero by construction (printed below), and
stretches without transients come out below zero. A late-ITI value below zero is therefore partly
set by this centering; the test cannot tell a tonic dip from it. Two comparisons outside the
criterion are printed for this: the late-ITI level against the level just before the cue (a
pre-cue baseline), and the short-ITI last bin against the long-ITI late-ITI level.

In [ ]:
L = T[T.cls == "long"]
Sh = T[T.cls == "short"]
rows = []
for corr in ("fit", "own"):
    for iti in ("reward", "cue"):
        for obs in ("trials", "samples"):
            col = f"tonic_{corr}_{iti}_{obs}"
            pm = L.groupby("subject")[col].mean()
            tt = stats.ttest_1samp(pm.dropna(), 0, alternative="less")
            last = Sh.groupby("subject")[f"last_{corr}_{obs}"].mean()
            rows.append({"reading": f"{corr}/{iti}/{obs}", "long_mean": pm.mean(), "t": tt.statistic,
                         "df": int(pm.notna().sum()) - 1, "p_one_tailed": tt.pvalue,
                         "mice_below_0": int((pm < 0).sum()),
                         "sessions_with_value": int(L[col].notna().sum()),
                         "trials_used": int(L[f"ntonic_{corr}_{iti}_{obs}"].sum()),
                         "short_last_bin_mean": last.mean(), "short_last_bin_sem": last.sem(),
                         "short_mice_above_0": int((last > 0).sum())})
fig4b = pd.DataFrame(rows)
print(fig4b.to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nlong-ITI sessions:", len(L), "| short-ITI sessions:", len(Sh))
for corr in ("fit", "own"):
    print(f"session mean ΔF/F ({corr} reading) over all samples: median", f"{T[f'dff_mean_{corr}'].median():.2g}",
          "| largest absolute value", f"{T[f'dff_mean_{corr}'].abs().max():.2g}")
print("last kept bin of the session-average trace in short sessions (s after reward, trial reading):",
      Sh.lastbin_fit_trials.value_counts().sort_index().to_dict())
print("\nPer mouse, reading fit/reward/trials:")
print(pd.DataFrame({"long_tonic": L.groupby("subject").tonic_fit_reward_trials.mean(),
                    "short_last_bin": Sh.groupby("subject").last_fit_trials.mean()}).round(5).to_string())
# against a pre-cue baseline in place of the criterion's zero: the cue-locked level 5 to 0 s before each cue
pre = cue_tr[[-5, -4, -3, -2, -1]].mean(1)
pre_L = pre[L.index].groupby(L.subject).mean()
diff_L = L.groupby("subject").tonic_fit_reward_trials.mean() - pre_L
tt = stats.ttest_1samp(diff_L, 0)
print("\nOther baseline, not the criterion's (fit/reward/trials): long-ITI pre-cue level (-5 to 0 s) per mouse, mean",
      f"{pre_L.mean():.4f}", "| late-ITI tonic minus pre-cue level: mean", f"{diff_L.mean():.4f}",
      f"t(17) = {tt.statistic:.2f}, two-tailed p = {tt.pvalue:.2g}, below in {(diff_L < 0).sum()} of 18")
last_S = Sh.groupby("subject").last_fit_trials.mean()
lt = L.groupby("subject").tonic_fit_reward_trials.mean()
tt = stats.ttest_rel(last_S, lt)
print("short-ITI last kept bin against the long-ITI late-ITI level, per mouse: mean", f"{last_S.mean():.4f}",
      "against", f"{lt.mean():.4f}", f"| paired t(17) = {tt.statistic:.2f}, p = {tt.pvalue:.2g}, short above long in",
      int((last_S > lt).sum()), "of 18")

In [ ]:
tr_long = np.vstack([s["trace_fit"] for s, c in zip(summaries, T.cls) if c == "long"])
tr_short = np.vstack([s["trace_fit"] for s, c in zip(summaries, T.cls) if c == "short"])
mL, mS = np.nanmean(tr_long, 0), np.nanmean(tr_short, 0)
fig, ax = plt.subplots(figsize=(6, 3))
ax.plot(np.arange(80) + 0.5, mL, label="long ITI")
ax.plot(np.arange(80) + 0.5, mS, label="short ITI")
ax.axhline(0, color="k", lw=0.5)
ax.set_xlabel("time from reward (s)"); ax.set_ylabel("ΔF/F (fit reading)"); ax.legend()
ax.set_ylim(-0.02, 0.05)
plt.show()
print("across-session mean ΔF/F in 1 s bins after reward, short ITI, bins 0-8:", np.round(mS[:9], 4))
print("long ITI, bins 0, 5, ..., 60:", np.round(mL[:61:5], 4))
for nm, m in (("long", mL), ("short", mS)):
    f = m[np.isfinite(m)]
    print(f"{nm}: plotted range {f.min():.4f} to {f.max():.4f}; bins above the axis limit {(f > 0.05).sum()}, below {(f < -0.02).sum()}")

**Result: partially reproduced.** In long-ITI sessions the late-ITI ΔF/F is below zero in all
18 mice under every reading (mean -0.0036 to -0.0061, t(17) from -11.6 to -13.7, one-tailed p
below 1e-9), and under the adopted reading from reward (mean -0.0046 and -0.0061). The short-ITI
part does not hold: the last kept bin before the next cue is below zero (mean -0.0067 under the
trial reading, -0.0056 under the sample reading, above zero in 1 and 0 of 18 mice), so the
short-ITI trace has fallen below the baseline of zero before the next trial. As the session mean
ΔF/F is zero by construction (median 2e-08 for the fit reading, 0 for the own reading), values
below zero in quiet stretches are expected from the correction alone; the long-ITI part matches
the paper's sign, but it does not by itself show a tonic dip. Measured against the level 5 to 0 s
before the cue, the late-ITI level shows no dip (difference 0.0003, t(17) = 1.03, p = 0.32,
below in 8 of 18 mice). Compared with the long-ITI late-ITI level, the short-ITI last bin is not
higher (-0.0067 against -0.0046, paired t(17) = -1.87, p = 0.078, higher in 5 of 18), so the
short-ITI part fails under that comparison too.

## fig4d-dip-deepens-with-training

> In long-ITI sessions the tonic dopamine dip deepens across training sessions (r = -0.29,
> p < 0.001).

In [ ]:
rows = []
for corr in ("fit", "own"):
    for iti in ("reward", "cue"):
        for obs in ("trials", "samples"):
            col = f"tonic_{corr}_{iti}_{obs}"
            for sn in ("sess_all", "sess_long"):
                m = ctd.mixed_slope(L.rename(columns={col: "y"}), "y", sn)
                rows.append({"tonic": f"{corr}/{iti}/{obs}", "session_number": sn, **m})
fig4d = pd.DataFrame(rows)
fig4d["r_pooled_in_tol"] = (fig4d.r_pooled - -0.29).abs() <= 0.029
fig4d["r_within_in_tol"] = (fig4d.r_within - -0.29).abs() <= 0.029
fig4d["reproduced"] = (fig4d.slope < 0) & (fig4d.p < 0.001) & (fig4d.r_pooled_in_tol | fig4d.r_within_in_tol)
fig4d["partial"] = (fig4d.slope < 0) & (fig4d.p < 0.05)
print(fig4d.drop(columns=["subjects"]).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
print("\nreadings reproduced:", int(fig4d.reproduced.sum()), "| slope negative with p < 0.05:",
      int(fig4d.partial.sum()), "of", len(fig4d), "| slope negative:", int((fig4d.slope < 0).sum()),
      "| p < 0.001:", int((fig4d.p < 0.001).sum()), "| r within tolerance (either r):",
      int((fig4d.r_pooled_in_tol | fig4d.r_within_in_tol).sum()))
print("models not converged:", int((~fig4d.converged).sum()))

In [ ]:
per_mouse = L.groupby("subject").apply(lambda g: pd.Series({
    "sessions": len(g), "slope_per_session": np.polyfit(g.sess_all, g.tonic_fit_reward_trials, 1)[0]
    if g.tonic_fit_reward_trials.notna().sum() > 2 else np.nan}))
print("Per mouse OLS slope of the tonic measure (fit/reward/trials) on session number:")
print(per_mouse.round(6).to_string())
print("negative in", int((per_mouse.slope_per_session < 0).sum()), "of", int(per_mouse.slope_per_session.notna().sum()))

**Result: partially reproduced.** The fixed slope of the tonic measure on session number is
negative under all 16 readings and below p = 0.05 under 10 of them, but never below p = 0.001
together with r within tolerance. Under the first-listed reading the r after removing each
mouse's mean is -0.28 (within tolerance of -0.29) and the slope's p is 0.0043; the pooled r is
-0.09 to -0.15 under every reading. The two readings with p below 0.001 (fit/cue/samples and
own/cue/samples, all sessions) have r -0.13 pooled and -0.35 within mice. Under the long-ITI
session numbering, the trial readings give p = 0.113 to 0.133.

## fig4c-lick-rate-rises-with-training

> Anticipatory lick rate increases across training sessions, in parallel with the deepening tonic
> dip (r = 0.65).

In [ ]:
rows = []
for sess, D in (("long only", L), ("all", T)):
    for sn in (("sess_all", "sess_long") if sess == "long only" else ("sess_all",)):
        rows.append({"sessions": sess, "session_number": sn, **ctd.mixed_slope(D, "lick_rate", sn)})
fig4c = pd.DataFrame(rows)
fig4c["r_pooled_in_tol"] = (fig4c.r_pooled - 0.65).abs() <= 0.065
fig4c["r_within_in_tol"] = (fig4c.r_within - 0.65).abs() <= 0.065
print(fig4c.drop(columns=["subjects"]).to_string(index=False, float_format=lambda x: f"{x:.4g}"))
pm = L.groupby("subject").apply(lambda g: np.polyfit(g.sess_all, g.lick_rate, 1)[0])
print("\nPer mouse OLS slope of long-ITI lick rate on session number (Hz per session): positive in",
      int((pm > 0).sum()), "of", len(pm), "| range", pm.min().round(3), "to", pm.max().round(3))

**Result: reproduced.** In long-ITI sessions the fixed slope of the lick rate on session number is
positive with p below 1e-4, and the pooled r is 0.65 (within tolerance of 0.65) under both
session numberings. Over all sessions r is 0.54, out of tolerance.

## krausz-included-observations

> The approach-epoch selection of the Krausz hex-level data keeps 264,042 of 298,918
> observations across 10 rats.

In [ ]:
steps = pd.Series({
    "all rows": len(hexdf),
    "at a port (port != -100)": int((hexdf.port != -100).sum()),
    "no distance to port (start hex of a trial)": int(hexdf.hexesFromPort.isna().sum()),
    "distance 0 (at the port)": int((hexdf.hexesFromPort == 0).sum()),
    "missing velocity": int(hexdf.vel.isna().sum()),
    "missing DA": int(hexdf.DA.isna().sum()),
    "rows after reward within a trial": int((hexdf.groupby(["session", "trial"]).rwd.cumsum() - hexdf.rwd > 0).sum()),
})
print(steps.to_string())
print("included:", int(inc.sum()), "| rats:", I.rat.nunique(), "| per rat:", I.rat.value_counts().sort_index().to_dict())

**Result: reproduced.** Dropping rows at a port, rows without a distance to the chosen port,
rows at distance 0 and rows without velocity keeps 264,042 rows across 10 rats, the paper's
count. No row follows a reward within its trial, so that condition removes nothing.

## fig2a-ramp-toward-goal

> In freely moving rats, NAc dopamine ramps up as the rats approach a reward port.

In [ ]:
pm = I.groupby(["rat", "d"]).DA.mean().reset_index()
r2a = ctd.per_rat_r(pm, "d", "DA")
tt = stats.ttest_1samp(r2a, 0)
print("per rat r(distance, mean DA):", r2a.round(3).to_dict())
print(f"mean r {r2a.mean():.3f}, t(9) = {tt.statistic:.2f}, p = {tt.pvalue:.2g}; negative in {(r2a < 0).sum()} of 10")
ph = I.groupby(["rat", "hexlabel"]).agg(DA=("DA", "mean"), d=("d", "mean")).reset_index()
r2a_hex = ctd.per_rat_r(ph, "d", "DA")
tt_hex = stats.ttest_1samp(r2a_hex, 0)
print(f"per hex position (mean DA and mean distance of each hex): mean r {r2a_hex.mean():.3f}, "
      f"t(9) = {tt_hex.statistic:.2f}, p = {tt_hex.pvalue:.2g}; negative in {(r2a_hex < 0).sum()} of 10")
curve = pm.groupby("d").DA.mean()
print("across-rat mean DA at distance 1, 5, 10, 15, 20:", curve.loc[[1, 5, 10, 15, 20]].round(3).to_dict())

**Result: reproduced.** DA is higher nearer the port: the per-rat r between distance and mean DA
per distance is negative in 9 of 10 rats (mean -0.68, t(9) = -7.25, p = 4.8e-05). Averaging per
hex position, as the Methods say, with each hex at its mean distance, the mean r is -0.30
(t(9) = -5.19, p = 0.00057), negative in 9 of 10 rats.

## fig2b-ramp-scales-with-reward

> The dopamine ramp is steeper on approaches to high reward (80 µL or more) than to low reward
> (20 µL or less).

In [ ]:
hi, lo = I[I.dest_rwd >= 80], I[I.dest_rwd <= 20]
s_hi = hi.groupby("rat").apply(lambda g: ctd.slope(g.d, g.DA))
s_lo = lo.groupby("rat").apply(lambda g: ctd.slope(g.d, g.DA))
m_hi, m_lo = hi.groupby("rat").DA.mean(), lo.groupby("rat").DA.mean()
for name, a, b in (("(a) slope of DA on distance", s_hi, s_lo), ("(b) mean DA", m_hi, m_lo)):
    tt = stats.ttest_rel(a, b)
    print(f"{name}: high {a.mean():.4f}, low {b.mean():.4f}, paired t(9) = {tt.statistic:.2f}, p = {tt.pvalue:.2g}")
print("rows high", len(hi), "low", len(lo))
print(pd.DataFrame({"slope_high": s_hi, "slope_low": s_lo, "mean_high": m_hi, "mean_low": m_lo}).round(4).to_string())

**Result: reproduced.** The slope of DA on distance is more negative for high-reward approaches
(-0.036 against -0.015 per hex, t(9) = -7.30, p = 4.6e-05), and mean DA is higher (p = 2.8e-05).

## fig2c-ramp-similar-early-late

> Ramps in early and late sessions look similar, with at most a modest reduction in ramp amplitude
> from early to late sessions.

First and last quartile of each rat's sessions in date order: the first and last
round(n/4) sessions, at least one (a standard choice; the Methods say only "first quartile" and
"last quartile").

In [ ]:
def early_late(g):
    ss = np.sort(g.session.unique()); k = max(1, int(round(len(ss) / 4)))
    e, l = g[g.session.isin(ss[:k])], g[g.session.isin(ss[-k:])]
    return pd.Series({"sessions": len(ss), "per_quartile": k, "early": ctd.slope(e.d, e.DA), "late": ctd.slope(l.d, l.DA)})
el = I.groupby("rat").apply(early_late)
tt = stats.ttest_rel(el.early, el.late)
print(el.round(4).to_string())
print(f"early slope {el.early.mean():.4f}, late {el.late.mean():.4f}, change {np.mean(el.late - el.early):.4f}; "
      f"paired t(9) = {tt.statistic:.2f}, p = {tt.pvalue:.3f}")
print("ramp amplitude as -slope: early", round(-el.early.mean(), 4), "late", round(-el.late.mean(), 4))

**Result: reproduced.** The paired t-test finds no difference between early and late ramp slopes
(-0.025 against -0.028, t(9) = 0.67, p = 0.52); the ramp is, if anything, slightly steeper late.

## fig2d-ramp-scales-with-velocity

> On fast approaches, dopamine is larger at the same positions than on slow ones.

In [ ]:
def fast_slow(g):
    lo_, hi_ = g.vel.quantile([0.25, 0.75])
    f, s = g[g.vel >= hi_].groupby("hexlabel").DA.mean(), g[g.vel <= lo_].groupby("hexlabel").DA.mean()
    com = f.index.intersection(s.index)
    return pd.Series({"positions": len(com), "fast": f[com].mean(), "slow": s[com].mean()})
fs = I.groupby("rat").apply(fast_slow)
tt = stats.ttest_rel(fs.fast, fs.slow)
print(fs.round(4).to_string())
print(f"fast {fs.fast.mean():.4f}, slow {fs.slow.mean():.4f}, paired t(9) = {tt.statistic:.2f}, p = {tt.pvalue:.2g}; "
      f"fast larger in {(fs.fast > fs.slow).sum()} of 10")

**Result: reproduced.** At the hex positions present in both splits, DA on fast approaches is
larger than on slow ones in 10 of 10 rats (t(9) = 7.40, p = 4.1e-05).

## fig2e-velocity-covaries-with-distance

> Velocity co-varies with distance to goal, and average dopamine and velocity covary across
> distance.

In [ ]:
pmv = I.groupby(["rat", "d"])[["DA", "vel"]].mean().reset_index()
r_dv, r_av = ctd.per_rat_r(pmv, "d", "vel"), ctd.per_rat_r(pmv, "DA", "vel")
for name, r in (("r(distance, velocity)", r_dv), ("r(mean DA, mean velocity)", r_av)):
    tt = stats.ttest_1samp(r, 0)
    print(f"{name}: mean {r.mean():.3f}, range {r.min():.3f} to {r.max():.3f}, t(9) = {tt.statistic:.2f}, p = {tt.pvalue:.2g}")
print("across-rat mean velocity (cm/s) at distance 1, 5, 10, 15, 20:",
      pmv.groupby("d").vel.mean().loc[[1, 5, 10, 15, 20]].round(1).to_dict())

**Result: reproduced.** Velocity is higher nearer the goal (mean per-rat r with distance -0.66,
p = 5.0e-05), and mean DA and velocity covary positively across distance (mean r 0.59,
p = 7.1e-06).

## fig2f-coupling-grows-near-goal

> The within-position coupling of velocity and dopamine increases with proximity to the goal
> (r = -0.494, df = 18, p = 0.027).

Under readings (a) and (b) the 20 distances are 1 to 20, the 20 nearest the goal; distance 21
has more rows than 20 (1097 against 908; see the counts printed), so the 20 distances with the
most rows (1 to 19 and 21) are shown under (a) as well. Under (c) the 20 hex positions with the
most included rows, each at its mean distance.

In [ ]:
print("included rows per distance:", I.d.value_counts().sort_index().astype(int).to_dict())
D = np.arange(1, 21)
ra = [stats.pearsonr(I[I.d == x].DA, I[I.d == x].vel)[0] for x in D]
rb = [np.nanmean([stats.pearsonr(g.DA, g.vel)[0] for _, g in I[I.d == x].groupby("rat") if len(g) > 2]) for x in D]
top = I.hexlabel.value_counts().index[:20]
rc = [stats.pearsonr(I[I.hexlabel == h].DA, I[I.hexlabel == h].vel)[0] for h in top]
dc = [I[I.hexlabel == h].d.mean() for h in top]
Dm = np.sort(I.d.value_counts().index[:20].to_numpy())
ram = [stats.pearsonr(I[I.d == x].DA, I[I.d == x].vel)[0] for x in Dm]
for name, x, y in (("(a) pooled r per distance", D, ra), ("(a) the same, 20 distances with most rows", Dm, ram),
                   ("(b) per-rat r averaged", D, rb), ("(c) 20 hex positions", dc, rc)):
    r = stats.pearsonr(x, y)
    print(f"{name}: r = {r.statistic:.3f}, df = {len(x) - 2}, p = {r.pvalue:.2g}; within tolerance "
          f"{abs(r.statistic - -0.494) <= 0.0494}; coupling range {min(y):.3f} to {max(y):.3f}")

**Result: reproduced.** Under reading (a), the within-distance DA-velocity r falls with distance
to the goal, r = -0.494 with df = 18 and p = 0.027, the paper's values, over distances 1 to 20.
Over the 20 distances with the most rows (1 to 19 and 21) the same reading gives r = -0.56
(p = 0.010), out of tolerance, so the match depends on taking the 20 distances nearest the goal.
Reading (b) gives r = -0.58 (p = 0.0075, out of tolerance), and reading (c), over hex positions,
the opposite sign.

## fig2gh-stationary-dip

> When rats are stationary (velocity below 5 cm/s), dopamine dips below zero (t = -4.27, df = 9,
> p = 0.001, one-tailed).

Reading (b), approach and non-approach observations, can be formed from the table in more than
one way; each is shown. Rows at a port are the rat at the port itself, before or after the reward.

In [ ]:
sets = {
    "(a) approach observations": I,
    "(b1) all rows off the ports": hexdf[hexdf.vel.notna() & (hexdf.port == -100)],
    "(b2) all rows, ports included": hexdf[hexdf.vel.notna()],
}
rows = []
for name, X in sets.items():
    st = X[X.vel < ctd.STATIONARY].groupby("rat").DA.mean()
    mv = X[X.vel >= ctd.MOVING].groupby("rat").DA.mean()
    tt = stats.ttest_1samp(st, 0, alternative="less")
    pr = stats.ttest_rel(mv, st)
    rows.append({"reading": name, "rows": len(X), "stationary_rows": int((X.vel < ctd.STATIONARY).sum()),
                 "mean_stationary_DA": st.mean(), "rats_below_0": int((st < 0).sum()), "t": tt.statistic,
                 "p_one_tailed": tt.pvalue, "t_in_tol": abs(tt.statistic - -4.27) <= 0.427,
                 "moving_minus_stationary": (mv - st).mean(), "paired_t": pr.statistic, "paired_p": pr.pvalue})
print(pd.DataFrame(rows).to_string(index=False, float_format=lambda x: f"{x:.4g}"))

**Result: reproduced, resting on an assumption.** Taking reading (b) as all rows off the ports,
stationary DA is below zero with t(9) = -4.27 and one-tailed p = 0.001, the paper's values. With
approach observations only, reading (a), the dip is weaker (t = -2.81, p = 0.010); with the port
rows included it is not significant (t = -1.46, p = 0.089). Moving DA exceeds stationary DA in the
paired test under (a) and (b1) (p = 0.012 and below).

## fig3-recovered-value-increases

> The value function recovered from moving minus stationary dopamine shows a net increase toward
> the goal in 8 of 10 rats.

The Methods give no minimum number of observations for a position's moving and stationary
means; two readings are run: at least 1 (no minimum, the literal reading) and at least 5 of each.
The Methods give Figure 2G-H and Figure 3 the same moving and stationary estimates, and Figure
2G-H matched the paper only with all rows off the ports (b1), so both row sets are run: (a) the
approach rows, and (b1) all rows off the ports. In (b1) the start hex of each trial has no
distance in the table; it is placed one hex beyond the trial's next row (without it, (b1) would
add only 290 rows at distance 0, which have no place on the axis). The profiles keep a positive
scale (decaying with distance), as the paper reads A > 0 as a net increase toward the goal; the
scale is searched from 1e-6 (σ from 0.1) to 20 per hex (hyperbolic 1000, σ 10,000 hexes), and fits
whose best grid point is at a bound are counted. At the upper σ bound and the lower rate bound a
profile is nearly quadratic or linear in distance, and the sign of A still gives the direction of
the fitted change.

In [ ]:
X1 = hexdf[hexdf.vel.notna() & (hexdf.port == -100)].copy()
nxt = X1.groupby(["session", "trial"]).d.shift(-1)
X1["d"] = X1.d.fillna(nxt + 1)
X1 = X1[X1.d >= 1]
fits3 = {}
rows = []
for rset, X in (("(a) approach", I), ("(b1) off the ports", X1)):
    for mo in (1, 5):
        for rat, g in X.groupby("rat"):
            d, V = ctd.recover_value(g, mo)
            f = {k: ctd.fit_profile(k, d, V) for k in ctd.PROFILES}
            w = min(f, key=lambda k: f[k]["bic"])
            fits3[(rset, mo, rat)] = (d, V, f)
            rows.append({"rows": rset, "min_obs": mo, "rat": rat, "distances": len(d), "V_near": V[0],
                         "winner": w, "A_winner": f[w]["A"], "scale_winner": f[w]["p"],
                         "winner_at_bound": f[w]["at_bound"], "fits_at_bound": sum(f[k]["at_bound"] for k in f),
                         **{f"r2_{k}": f[k]["r2"] for k in f}})
fig3 = pd.DataFrame(rows)
print("rows (b1):", len(X1), "| stationary rows (a)", int((I.vel < ctd.STATIONARY).sum()),
      "(b1)", int((X1.vel < ctd.STATIONARY).sum()), "| V is 0 at the farthest distance")
print(fig3.to_string(index=False, float_format=lambda x: f"{x:.3g}"))
for (rset, mo), g in fig3.groupby(["rows", "min_obs"]):
    print(f"\n{rset}, min_obs {mo}: rats with A > 0 in the BIC winner: {(g.A_winner > 0).sum()} of 10; "
          f"V nearest above V farthest: {(g.V_near > 0).sum()} of 10; winners {g.winner.value_counts().to_dict()}; "
          f"winners at a bound {int(g.winner_at_bound.sum())}")
    print("   mean R² by shape:", g[[c for c in g if c.startswith("r2_")]].mean().round(3).to_dict())
narrow3 = []
for rset, X in (("(a) approach", I), ("(b1) off the ports", X1)):
    for mo in (1, 5):
        for rat in sorted(X.rat.unique()):
            d, V, _ = fits3[(rset, mo, rat)]
            f = {k: ctd.fit_profile(k, d, V, ctd.PROFILE_BOUNDS_NARROW) for k in ctd.PROFILES}
            w = min(f, key=lambda k: f[k]["bic"])
            narrow3.append({"rows": rset, "min_obs": mo, "winner": w, "A_pos": f[w]["A"] > 0})
narrow3 = pd.DataFrame(narrow3)
print("\nFirst round's narrower profile bounds", ctd.PROFILE_BOUNDS_NARROW)
for (rset, mo), g in narrow3.groupby(["rows", "min_obs"]):
    print(f"   {rset}, min_obs {mo}: rats with A > 0 in the BIC winner: {int(g.A_pos.sum())} of 10; "
          f"winners {g.winner.value_counts().to_dict()}")

**Result: partially reproduced, resting on an assumption.** With the approach rows and no
minimum number of observations, the BIC-winning profile has A > 0 in 7 of 10 rats, against 8 in
the paper; with at least 5 observations per mean it is 6 of 10. With all rows off the ports (b1),
the row set fig2gh adopts, the counts are the same, 7 and 6, so the row set does not bring the
count to 8; the approach rows are recorded, as the Methods name them. With the first round's
narrower bounds of the profile scale the counts and winners are the same (printed last). Five of
the ten winning fits with the approach rows and no minimum sit at the upper σ bound, where the
Gaussian is nearly quadratic in distance, so where the Gaussian wins there it partly wins as a
quadratic against nearly linear shapes. Under the minimum of 5, the IM-1292 fit has A = -2.6e11
at σ = 0.78 with the approach rows (-5.2e10 at σ = 0.80 with b1): the Gaussian there fits only
the nearest point, which lies below the rest. IM-1532's winning power law with the approach rows
sits at the lower bound k = 1e-6 with A = 2.1e4, nearly linear in distance.

## fig3cd-gaussian-shape-wins

> All four candidate shapes fit each rat's recovered value function well (mean R² above 0.8), and
> the best shape varies across rats, Gaussian in 8, exponential in 1 and power-law in 1.

In [ ]:
for (rset, mo), g in fig3.groupby(["rows", "min_obs"]):
    r2 = g[[c for c in g if c.startswith("r2_")]].mean()
    print(f"{rset}, min_obs {mo}: winners {g.winner.value_counts().to_dict()}; mean R² {r2.round(3).to_dict()}; "
          f"all four above 0.8: {bool((r2 > 0.8).all())}")

**Result: partially reproduced.** With the approach rows the Gaussian profile wins in 9 of 10
rats under both readings (the paper: 8, with 1 exponential and 1 power-law); the tenth is
exponential with no minimum and power-law with a minimum of 5. With all rows off the ports it wins
in 10 of 10 with no minimum, and in 8 of 10, with 1 exponential and 1 hyperbolic, with a minimum
of 5. The mean R² is above 0.8 for all four shapes under no reading: with no minimum it is 0.50 to
0.73; with a minimum of 5, three of four shapes exceed 0.8 with the approach rows (hyperbolic
0.77) and two of four with all rows off the ports.

## fig5bc-gaussian-ct-td-best

> Of six models of approach dopamine, the Gaussian continuous-time TD model fits best summed
> across rats and wins in 8 of 10 rats by BIC.

The Methods fit each model by Nelder-Mead from many starts. Here the linear parameters (A, B, c)
are solved by least squares for each value of the spatial scale, and the scale is found by a grid
and a bounded refinement. As Nelder-Mead is unconstrained, λ is searched from -5 to 5 per hex
(negative λ is a profile that rises with distance) and σ from 0.3 to 10,000 hexes (flat over the
maze). A Nelder-Mead search over all parameters is then run from each optimum to check that it
finds nothing lower. The first round's narrower search (λ from 1e-4 to 5, σ from 0.3 to 100) is
shown beside it.

In [ ]:
t0 = time.time()
narrow = {"exp": np.geomspace(1e-4, 5, 120), "gauss": np.geomspace(0.3, 100, 120)}
B, B0, S, S0, P, P0, NM = {}, {}, {}, {}, {}, {}, {}
for rat, g in I.groupby("rat"):
    d, v, y = g.d.to_numpy(float), g.vel.to_numpy(), g.DA.to_numpy()
    fits = {m: ctd.fit_model(m, d, v, y) for m in ctd.MODEL_K}
    fits0 = {m: ctd.fit_model(m, d, v, y, grid=None if m == "Velocity" else narrow["gauss" if "gauss" in m else "exp"])
             for m in ctd.MODEL_K}
    B[rat] = {m: f["bic"] for m, f in fits.items()}
    B0[rat] = {m: f["bic"] for m, f in fits0.items()}
    S[rat] = {m: f["sse"] for m, f in fits.items()}
    S0[rat] = {m: f["sse"] for m, f in fits0.items()}
    P0[rat] = {m: f.get("p") for m, f in fits0.items()}
    P[rat] = {m: f.get("p") for m, f in fits.items()}
    P[rat]["at_bound"] = sum(bool(f.get("at_bound", False)) for f in fits.values())
    NM[rat] = {m: ctd.nelder_mead_check(m, d, v, y, f) - f["sse"] for m, f in fits.items() if m != "Velocity"}
B, B0, S, S0, P, P0, NM = (pd.DataFrame(x).T for x in (B, B0, S, S0, P, P0, NM))
summed, summed0 = B.sum(), B0.sum()
print("SSE per rat and model:")
print(S.round(2).to_string())
print("\nΔBIC per rat relative to its best model:")
print((B.sub(B.min(1), axis=0)).round(1).to_string())
print("\nsummed BIC minus the best summed:", (summed - summed.min()).round(1).to_dict())
print("winners:", B.idxmin(1).value_counts().to_dict())
print("first round's narrower search: summed BIC minus the best", (summed0 - summed0.min()).round(1).to_dict(),
      "| winners", B0.idxmin(1).value_counts().to_dict())
print("\nfitted spatial scale per rat (λ per hex or σ in hexes), and fits whose best grid point was at a bound:")
print(P.round(3).to_string())
dS = (S0 - S).stack()
print("\nSSE of the narrower search minus the wide one, fits lowered by more than 0.01:")
print(dS[dS > 0.01].round(2).to_string())
print("IM-1322 CT TD gauss: SSE", round(S.loc["IM-1322", "CT TD gauss"], 2), "at σ", round(P.loc["IM-1322", "CT TD gauss"], 1),
      "| narrower search", round(S0.loc["IM-1322", "CT TD gauss"], 2), "at σ", round(P0.loc["IM-1322", "CT TD gauss"], 1))
print("\nNelder-Mead from the optimum, change in SSE: lowest", f"{NM.min().min():.2g}", "| largest", f"{NM.max().max():.2g}")
print(f"fits took {time.time() - t0:.0f} s")

**Result: reproduced.** The Gaussian CT TD model has the lowest summed BIC and wins in 8 of 10
rats; dV/dt Gaussian wins in the other 2. With λ allowed below zero, the exponential CT TD and
Value fits of three rats (IM-1322, IM-1398, IM-1532) move to a negative λ and lower their SSE by
0.53 to 20.93, and the exponential CT TD model is now 412 above the Gaussian one in summed BIC
(473 with the first round's narrower search); the winners do not change. One Gaussian CT TD fit
(IM-1322) runs to the upper σ bound, where its SSE (11621.02) is 0.17 below the fit at σ = 100.
Nelder-Mead over all parameters from each optimum lowers no SSE by more than 1e-7.

## fig5d-tonic-term-and-gaussian-contribute

> In a factorial decomposition, the tonic term improves fit by ΔBIC = 1156 and the Gaussian
> spatial profile by ΔBIC = 706.

In [ ]:
readings = lambda s: {
    "(a) main effects": ((s["dV/dt exp"] - s["CT TD exp"] + s["dV/dt gauss"] - s["CT TD gauss"]) / 2,
                         (s["CT TD exp"] - s["CT TD gauss"] + s["dV/dt exp"] - s["dV/dt gauss"]) / 2),
    "(b) tonic at Gaussian, profile at CT TD": (s["dV/dt gauss"] - s["CT TD gauss"], s["CT TD exp"] - s["CT TD gauss"]),
    "(c) tonic at exponential, profile at dV/dt": (s["dV/dt exp"] - s["CT TD exp"], s["dV/dt exp"] - s["dV/dt gauss"]),
}
for label, s in (("wide search", summed), ("first round's narrower search", summed0)):
    print(label)
    for k, (tonic, prof) in readings(s).items():
        print(f"  {k}: tonic ΔBIC {tonic:.1f} (1156 ± 115.6: {abs(tonic - 1156) <= 115.6}), "
              f"Gaussian profile ΔBIC {prof:.1f} (706 ± 70.6: {abs(prof - 706) <= 70.6})")

**Result: reproduced.** Under reading (a), the main effects, the tonic term improves the summed
BIC by 1201 and the Gaussian profile by 665, both within tolerance of 1156 and 706. With the
first round's narrower search they were 1170 and 696; allowing λ below zero moved both by 31,
and both stay within the tolerance.

## Summary

| Claim | Data | Outcome |
|---|---|---|
| floeder-subjects-sessions | DANDI:001552 | reproduced |
| fig4a-lick-short-above-long | DANDI:001552 | reproduced |
| fig4b-long-iti-dip-below-baseline | DANDI:001552 | partially reproduced |
| fig4d-dip-deepens-with-training | DANDI:001552 | partially reproduced |
| fig4c-lick-rate-rises-with-training | DANDI:001552 | reproduced |
| krausz-included-observations | hexLevelDf (elsewhere) | reproduced |
| fig2a-ramp-toward-goal | hexLevelDf (elsewhere) | reproduced |
| fig2b-ramp-scales-with-reward | hexLevelDf (elsewhere) | reproduced |
| fig2c-ramp-similar-early-late | hexLevelDf (elsewhere) | reproduced |
| fig2d-ramp-scales-with-velocity | hexLevelDf (elsewhere) | reproduced |
| fig2e-velocity-covaries-with-distance | hexLevelDf (elsewhere) | reproduced |
| fig2f-coupling-grows-near-goal | hexLevelDf (elsewhere) | reproduced |
| fig2gh-stationary-dip | hexLevelDf (elsewhere) | reproduced, resting on an assumption |
| fig3-recovered-value-increases | hexLevelDf (elsewhere) | partially reproduced, resting on an assumption |
| fig3cd-gaussian-shape-wins | hexLevelDf (elsewhere) | partially reproduced |
| fig5bc-gaussian-ct-td-best | hexLevelDf (elsewhere) | reproduced |
| fig5d-tonic-term-and-gaussian-contribute | hexLevelDf (elsewhere) | reproduced |
| figs2-no-velocity-coupling-without-goal | Brown et al. (2026) | blocked (methods underspecified) |

A failure to reproduce here may be a failure of this attempt, not of the paper: see the
reasoning and the differences from the paper's method.

**How the analysis differs from the paper's.** The photometry is placed on the event clock by a
fit of TTL2 edges to cue onsets, as the files' stated rates are wrong in part of the sessions.
Where the paper leaves a step open (the ΔF/F correction, where the ITI is measured from, what an
observation is in the 20-observation rule, session numbering, how r is computed, which rows form
"non-approach" observations, the minimum count per position in the value recovery, and which rows give the stationary
estimates of the value recovery), every reading is run and shown; for Figure 4B the ITI from
reward is adopted, as its medians (7.6 and 44.7 s) match the paper's ~8 and ~45 s. Two recordings deposited twice are counted once. The model fits of
Figure 5 solve the linear parameters by least squares and search the spatial scale on a wide grid
(λ from -5 to 5, σ up to 10,000 hexes), in place of multi-start Nelder-Mead, and a Nelder-Mead
search from each optimum confirms it; the mixed models of Figure 4 are fitted with three
optimizers and the fit with the highest REML likelihood is kept; the value recovery integrates with scipy's RK45 in place of
Julia's Tsit5. Baseline for Figure 4B is ΔF/F of zero, and the session-wide correction makes the
session mean ΔF/F zero, so a late-ITI value below zero is partly set by the correction.

**What was found about the dataset.** The dandiset holds the 18 mice of the paper with
photometry and events, and 9 more mice (`JF-VR-FP`) with photometry only and no event log or task
description. Two sessions are deposited twice under different days (`JF-dLight-ramp2-F2` Day3 and
Day4, `JF-dLight-ramp2-F4` Day5 and Day6). The photometry series of part of the `JF-FP` sessions
state a rate of 120481.9 Hz (or timestamps 8.3 microseconds apart) for a signal sampled at about
120.4 Hz, and in every file the photometry states a start of 0 on a clock that is not the event
log's; the only link between the two is the TTL2 channel, described only as "raw photometry
channel ttl2", whose rising edges fall on the cues.
The files hold no depositor ΔF/F (`processing/` has only the task parameters). Sessions are in
blocks, 8 long-ITI then 8 short-ITI sessions in `JF-FP`, and 7 or 8 long, 8 short, then 7 or 8
long in `JF-dLight-ramp2`, not alternating across days as the paper describes. In the Krausz table the nominal rewards the paper calls µL are reward
probabilities in percent, as the Krausz et al. code reads them. Short-ITI sessions have a median inter-cue interval of about
16.6 s and long ones about 53.7 s.

In [ ]:
print(f"notebook ran in {time.time() - started:.0f} s")